# NER LLM Prompts
This pipeline uses an LLM (either local or from HF) to perform NER using system prompts

In [ ]:
%load_ext autoreload 
%autoreload 2

import os

import torch
from dotenv import load_dotenv
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

load_dotenv()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

token = os.environ["HF_TOKEN"]

# model_name = "deepseek-ai/deepseek-llm-7b-base"
model_name = "deepseek-ai/deepseek-llm-7b-chat"
# model_name = "meta-llama/Llama-3.1-8B"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "google/gemma-7b"
# model_name = "google/gemma-7b-it"

# Finetuned
# model_name = "../src/deepseek-7b-llm-base-legal"
# model_name = "../models/meta-llama/Llama-3.1-8B_legal_parallel_chat"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,
    token=token,
)
# model.generation_config = GenerationConfig.from_pretrained(model_name)
# model.generation_config.pad_token_id = model.generation_config.eos_token_id


In [ ]:
# Query model
def query_model(prompt, max_new_tokens):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    output = model.generate(input_ids, max_new_tokens=max_new_tokens)
    response = tokenizer.decode(output[0], skip_special_tokens=True)

    return response


prompt = """
Effettua Named Entity Recognition sul testo in input. Restituisci una lista di tuple del tipo (token, tag NER). Riconosci soltanto PER, ORG, LOC e MISC. Termina l'output dopo la lista.

Per esempio:

Input: Rivafosca e dalla documentazione acquisita si evince che: il Nicola Romano gestì nei primi anni '60 un distributore di carburanti unitamente a Andrea Moretti; la deliberazione della Comune di Valleverde con cui venne accolta, per la parte di competenza del Agenzia Regionale per i Servizi, la domanda tendente ad ottenere la concessione per l’impianto dei distributori di carburante da collocare sul suolo pubblico di Via del Mercato all’angolo con Piazza delle Magnolie, fu adottata il 9 dicembre 1961, mentre era Sindaco Antonio Ferraro (in una adunanza della Consiglio Direttivo presieduta dal Vice Sindaco avv. Dario Pellegrini, con la presenza degli assessori effettivi Enrico Fontana, Simone Costa, Giorgio Lombardi, Fabio Mancini, Michele Rizzo, e degli assessori supplenti Giulia Bianchi Giulia Bianchi, Roberto Caruso, Claudio Serra, come si evince dalla documentazione acquisita all’udienza del 22 maggio 1996).
Output:
Rivafosca|PER
Nicola Romano|PER
Andrea Moretti|PER
Comune di Valleverde|ORG
Agenzia Regionale per i Servizi|ORG
Piazza delle Magnolie|LOC
Antonio Ferraro|PER
Dario Pellegrini|PER
Enrico Fontana|PER
Simone Costa|PER
Giorgio Lombardi|PER
Fabio Mancini|PER
Michele Rizzo|PER
Giulia Bianchi Giulia Bianchi|PER
Roberto Caruso|PER
Claudio Serra|PER
"""


def create_query(text: str) -> str:
    sent = f"Input: {text}\nOutput:"

    return prompt + sent


query = create_query(
    "Il Riccardo Galli ha specificato di aver conosciuto il Paride Valli presso la macelleria, sita nel quartiere Salamini, dei propri cugini Ettore Vitale, Ottavio Neri ed Leandro Sala (l’altro collaborante già citato), frequentata dal padre Riccardo Galli e da altri uomini d’altri tempi (come Gambino Simone Costa) e che il Paride Valli era persona vicina ed amica di Tullio Conte Di Napoli, circostanza anche personalmente constatata dal collaboratore allorché, in alcune occasioni, aveva incontrato il Paride Valli in compagnia del Di Napoli nella villa di proprietà di Enrico Fontana Gino Basile, altro uomo d’altri tempi della famiglia di Orlando Pace, nonché parente dello stesso Paride Valli ove lo stesso Di Napoli dimorava durante la sua gita."
)

response = query_model(query, max_new_tokens=300)

print(response[len(prompt) :])

In [ ]:
response[len(prompt) :].split("\n\n")[0].split("\n")[1][8:]

In [ ]:
import json
from datetime import datetime

from tqdm import tqdm

# Load jsonl file into a list of dictionaries
with open("../data/labelled/test.jsonl") as test_f:
    test_data = [json.loads(line) for line in test_f]


output = []
for sent in tqdm(test_data):
    query = create_query(sent["text"])
    response = query_model(query, max_new_tokens=300)

    output.append(response[len(prompt) :].split("\n\n")[1][8:])


In [ ]:
import itertools

pairs = [list(y) for x, y in itertools.groupby(output, lambda z: z == "\n") if not x][0]

clean_pairs = []
for p in pairs:
    lines = p.split("\n")
    output_pairs = []
    for l in lines:
        output_pairs.append(l.split("|"))

    clean_pairs.append(output_pairs)

In [ ]:
# Create timestamp to create unique output files
dt = str(datetime.now()).split(".")[0].replace(" ", "_").replace(":", "-")

# Prevent weird issues with paths
model_name = model_name.replace("../", "")

# Save predictions to file
output_file = f"../output/llm_ner_pred_{model_name.split('/')[-1]}_{dt}.txt"
with open(output_file, "w") as test_f:
    for line in output:
        test_f.writelines(line + "\n")

## Evaluation

### Span-based

In [ ]:
import json

# pred_filename = "llm_ner_pred_gemma-7b_2025-04-29_12-46-43"
# pred_filename = "llm_ner_pred_Llama-3.1-8B_2025-04-29_11-31-28"
# pred_filename = "llm_ner_pred_deepseek-llm-7b-base_2025-04-28_14-54-20"
# pred_filename = "llm_ner_pred_gemma-7b-it_2025-05-30_11-24-33"
# pred_filename = "llm_ner_pred_deepseek-llm-7b-chat_2025-05-30_11-54-59"
pred_filename = output_file.split("/")[-1].replace(".txt", "")


tmp_output_file = f"../output/{pred_filename}.txt"
tmp_output_file

In [ ]:
import json

from util import evaluate_ner

with open(f"../output/{pred_filename}_eval.json") as file:
    pred = json.load(file)

with open("../data/labelled/test_combined.jsonl") as file:
    true = [json.loads(line) for line in file.readlines()]

pred_lines = [p["label"] for p in pred]
true_lines = [t["label"] for t in true]

# Append index to each label to ensure unique elements when lists have been flattened
for i, line in enumerate(true_lines):
    for label in line:
        label.append(i)

for i, line in enumerate(pred_lines):
    for label in line:
        label.append(i)

In [ ]:
from itertools import chain

# Flatten lists to evaluate metrics on whole dataset
flat_preds = list(map(tuple, chain.from_iterable(pred_lines)))
flat_golds = list(map(tuple, chain.from_iterable(true_lines)))

results = evaluate_ner(flat_preds, flat_golds)
results

In [ ]:
with open("../output/experiments_results.json") as file:
    res = json.load(file)

res[pred_filename] = results

# Append results to json
with open("../output/experiments_results.json", "w") as file:
    json.dump(res, file, indent=4)